# Portfolio Optimization with Mean-Variance Analysis

This notebook demonstrates how to build an optimal investment portfolio using modern portfolio theory. We will:
1. Define functions for fetching historical data.
2. Implement a `PortfolioOptimizer` class for Mean-Variance Optimization.
3. Fetch data for a diversified set of Canadian-listed assets (including **CDRs** and **TSX ETFs**).
4. Optimize for the **Maximum Sharpe Ratio** and **Minimum Volatility**.
5. Visualize the **Efficient Frontier**.

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import datetime
from scipy.optimize import minimize
import plotly.graph_objects as go
import plotly.express as px

## 1. Core Logic: Data Loading and Optimization

First, let's define our core functions directly in the notebook for better intuition. 

### Data Loader Function
This function fetches historical prices using `yfinance`. It now supports a `market` parameter for a default suffix, but also respects existing suffixes like `.NE` used by CDRs.

In [2]:
def fetch_historical_prices(tickers, years=5, market=None):
    # Handle market suffix if provided (e.g., 'TO' for TSX)
    processed_tickers = []
    for t in tickers:
        # If the ticker already has a suffix (like .NE or .TO), use it as is
        if "." in t:
            processed_tickers.append(t)
        elif market:
            processed_tickers.append(f"{t}.{market}")
        else:
            processed_tickers.append(t)
            
    end_date = datetime.datetime.now()
    start_date = end_date - datetime.timedelta(days=years*365)
    
    # Fetch data
    data = yf.download(processed_tickers, start=start_date, end=end_date)
    
    # Handle MultiIndex and price naming
    if 'Adj Close' in data.columns.levels[0]:
        data = data['Adj Close']
    else:
        data = data['Close']
    
    # Clean data
    data = data.dropna(axis=0, how='all')
    data = data.ffill().bfill()
    
    return data

### Portfolio Optimizer Class
This class calculates portfolio performance (returns, volatility, and Sharpe ratio) and uses `scipy.optimize` to find the best asset weights.

In [3]:
class PortfolioOptimizer:
    def __init__(self, prices, risk_free_rate=0.02):
        self.prices = prices
        self.tickers = prices.columns
        self.risk_free_rate = risk_free_rate
        
        # Calculate daily returns
        self.returns = prices.pct_change().dropna()
        
        # Annualize returns and covariance (assuming 252 trading days)
        self.mean_returns = self.returns.mean() * 252
        self.cov_matrix = self.returns.cov() * 252
        
    def portfolio_performance(self, weights):
        # Returns: weighted sum of mean returns
        returns = np.sum(self.mean_returns * weights)
        # Risk (Volatility): square root of (weights * cov_matrix * weights_transposed)
        std = np.sqrt(np.dot(weights.T, np.dot(self.cov_matrix, weights)))
        # Sharpe Ratio: excess return per unit of risk
        sharpe = (returns - self.risk_free_rate) / std
        return returns, std, sharpe

    def _neg_sharpe_ratio(self, weights):
        return -self.portfolio_performance(weights)[2]

    def _volatility(self, weights):
        return self.portfolio_performance(weights)[1]

    def optimize_maximum_sharpe(self):
        num_assets = len(self.tickers)
        constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
        bounds = tuple((0, 1) for _ in range(num_assets))
        initial_guess = num_assets * [1. / num_assets,]
        
        result = minimize(self._neg_sharpe_ratio, initial_guess, 
                          method='SLSQP', bounds=bounds, constraints=constraints)
        return result.x

    def optimize_minimum_volatility(self):
        num_assets = len(self.tickers)
        constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
        bounds = tuple((0, 1) for _ in range(num_assets))
        initial_guess = num_assets * [1. / num_assets,]
        
        result = minimize(self._volatility, initial_guess, 
                          method='SLSQP', bounds=bounds, constraints=constraints)
        return result.x

    def get_efficient_frontier(self, target_returns):
        num_assets = len(self.tickers)
        efficient_frontier = []
        for target in target_returns:
            constraints = (
                {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},
                {'type': 'eq', 'fun': lambda x: self.portfolio_performance(x)[0] - target}
            )
            bounds = tuple((0, 1) for _ in range(num_assets))
            initial_guess = num_assets * [1. / num_assets,]
            result = minimize(self._volatility, initial_guess, 
                              method='SLSQP', bounds=bounds, constraints=constraints)
            if result.success:
                efficient_frontier.append(result.fun)
            else:
                efficient_frontier.append(None)
        return efficient_frontier

## 2. Load Data (TSX & CDR Portfolio)
We use a mix of **Canadian Depositary Receipts (CDRs)** and **TSX-listed ETFs**. 
- **CDRs** trade on **Cboe Canada** (formerly NEO) and use the `.NE` suffix.
- **TSX ETFs** use the `.TO` suffix.

*Note: CDRs are CAD-hedged, which removes currency risk when investing in US-based companies.*

In [4]:
# CDRs use .NE on Cboe Canada
# Note: The ticker for NIKE CDR is NKE.NE (not NIKE.NE)

tickers = [
    'AMD.NE', 'BNS.TO', 'COST.NE', 'GOOG.NE', 'NKE.NE', 
    'NVDA.NE', 'QQCC.TO', 'XEG.TO', 'XFN.TO', 'ZGD.TO', 'ZMIC.NE'
]

prices = fetch_historical_prices(tickers, years=3)
prices.head()

/var/folders/rp/v0tx4k4n1v9013rjc6zppvw80000gn/T/ipykernel_5723/2316474474.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(processed_tickers, start=start_date, end=end_date)
[*********************100%***********************]  11 of 11 completed


Ticker,AMD.NE,BNS.TO,COST.NE,GOOG.NE,NKE.NE,NVDA.NE,QQCC.TO,XEG.TO,XFN.TO,ZGD.TO,ZMIC.NE
Date,,,,,,,,,,,
2023-04-03,18.850000,57.209671,22.637457,17.685385,21.926765,6.706622,7.291130,13.963209,40.472282,84.063789,12.564713
2023-04-04,18.719999,56.929558,22.675858,17.734978,22.342619,6.584229,7.298290,13.846772,40.173901,86.461166,12.564713
2023-04-05,18.040001,57.294552,22.656660,17.744896,21.822803,6.441854,7.269641,13.757208,40.065376,86.347000,12.564713
2023-04-06,18.030001,57.277580,22.138241,18.379705,21.662136,6.486814,7.334102,13.649730,40.092514,87.000832,12.564713
2023-04-10,18.629999,57.387917,22.455050,18.052382,21.964569,6.614201,7.348427,13.757208,40.246216,86.398895,12.564713


## 3. Explain Sharpe Ratio Derivation
The Sharpe Ratio is calculated as: **(Expected Portfolio Return - Risk-Free Rate) / Portfolio Volatility**

Here's how we get each element:
1. **Expected Return**: The mean of historical daily returns, multiplied by the weights and annualized (x252).
2. **Risk-Free Rate**: Assumed to be 2% (0.02) annually (a standard benchmark).
3. **Volatility**: Derived from the covariance matrix of assets and the weights to calculate the annualized standard deviation.

## 4. How we get the Weights (Optimization)

We don't pick the weights by hand. We use a **Mathematical Optimizer** (`scipy.optimize.minimize`) to find them. 

### Component of the Optimization:
1. **Objective**: Minimize a cost function (either the *negative Sharpe Ratio* or *Volatility*).
2. **Initial Guess**: We start with equal weights for all stocks.
3. **Constraints**: 
    * `np.sum(x) - 1 == 0`: The weights MUST sum to 100%.
    * `bounds = (0, 1)`: We don't allow short selling (no negative weights).
4. **Algorithm**: We use **SLSQP** (Sequential Least Squares Programming).

In [5]:
optimizer = PortfolioOptimizer(prices)

max_sharpe_weights = optimizer.optimize_maximum_sharpe()
min_vol_weights = optimizer.optimize_minimum_volatility()

ret_ms, vol_ms, sharpe_ms = optimizer.portfolio_performance(max_sharpe_weights)
ret_mv, vol_mv, sharpe_mv = optimizer.portfolio_performance(min_vol_weights)

print(f"Max Sharpe: Return={ret_ms:.2%}, Volatility={vol_ms:.2%}, Sharpe={sharpe_ms:.2f}")
print(f"Min Vol: Return={ret_mv:.2%}, Volatility={vol_mv:.2%}, Sharpe={sharpe_mv:.2f}")

Max Sharpe: Return=32.24%, Volatility=13.49%, Sharpe=2.24
Min Vol: Return=20.89%, Volatility=10.49%, Sharpe=1.80


## 5. Visualize Efficient Frontier
The **Efficient Frontier** represents the set of optimal portfolios that offer the highest return for each level of risk.

In [6]:
# Generate Efficient Frontier points
target_returns = np.linspace(ret_mv, max(optimizer.mean_returns), 20)
efficient_vols = optimizer.get_efficient_frontier(target_returns)

# Sample 1000 random portfolios for comparison
n_random = 1000
random_results = []
for _ in range(n_random):
    w = np.random.random(len(tickers))
    w /= np.sum(w)
    random_results.append(optimizer.portfolio_performance(w))

rand_rets, rand_vols, rand_sharpes = zip(*random_results)

# Map to Plotly Scatter
fig = go.Figure()

# Random Portfolios plot
fig.add_trace(go.Scatter(
    x=rand_vols, y=rand_rets, mode='markers', 
    marker=dict(color=rand_sharpes, colorscale='Viridis', size=5, showscale=True, colorbar=dict(title="Sharpe Ratio")),
    name='Random Portfolios'
))

# Efficient Frontier line
fig.add_trace(go.Scatter(
    x=efficient_vols, y=target_returns, mode='lines', 
    line=dict(color='black', width=3, dash='dash'),
    name='Efficient Frontier'
))

# Target Points (Max Sharpe & Min Vol)
fig.add_trace(go.Scatter(
    x=[vol_ms], y=[ret_ms], mode='markers',
    marker=dict(color='red', size=15, symbol='star'),
    name='Max Sharpe Ratio'
))
fig.add_trace(go.Scatter(
    x=[vol_mv], y=[ret_mv], mode='markers',
    marker=dict(color='green', size=15, symbol='star'),
    name='Min Volatility'
))

fig.update_layout(
    title="Efficient Frontier & Portfolio Optimization",
    xaxis_title="Risk (Annual Volatility)",
    yaxis_title="Annual Return",
    template="plotly_white"
)
fig.show()

## 6. Asset Allocation: Max Sharpe vs. Min Volatility
Comparing the allocation of the high-growth strategy vs. the low-risk strategy.

In [7]:
# Max Sharpe Weights
weight_df_ms = pd.DataFrame({
    'Ticker': prices.columns,
    'Weight': max_sharpe_weights
}).sort_values('Weight', ascending=False)

fig_ms = px.pie(weight_df_ms, values='Weight', names='Ticker', 
             title='Max Sharpe Ratio: Optimal Allocation')
fig_ms.show()

# Min Volatility Weights
weight_df_mv = pd.DataFrame({
    'Ticker': prices.columns,
    'Weight': min_vol_weights
}).sort_values('Weight', ascending=False)

fig_mv = px.pie(weight_df_mv, values='Weight', names='Ticker', 
             title='Minimum Volatility: Optimal Allocation')
fig_mv.show()